# Go2 Track 2 Bonus Project: GPU Server Starter Notebook

Use this notebook from the `/home/shashank/RobotLocomotion-Race` checkout on the GPU server to inspect the interfaces, install dependencies, optionally train or reuse a low-level checkpoint, run single-policy track evaluation, and prepare submission metadata.

It starts from a weak baseline. Your leaderboard submission should train a learned high-level planner for the fixed 5D -> [vx, vy, yaw_rate] interface.


## 1. Configure server paths

This version runs against the existing repository cloned at `/home/shashank/RobotLocomotion-Race`. Set `COURSE_REPO_DIR` manually only if you move the checkout or launch Jupyter from a different server path.


In [ ]:
from pathlib import Path
import io
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from urllib.parse import urlparse

TEAM_NAME = "change_me"

# GPU server checkout path.
# You can override it before running this cell with:
#   os.environ["COURSE_REPO_DIR"] = "/path/to/RobotLocomotion-Race"
DEFAULT_SERVER_REPO_DIR = Path("/home/shashank/RobotLocomotion-Race")


def find_course_repo() -> Path:
    candidates = []
    env_dir = os.environ.get("COURSE_REPO_DIR")
    if env_dir:
        candidates.append(Path(env_dir).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents, DEFAULT_SERVER_REPO_DIR])
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "run_track_bonus.py").exists() and (candidate / "configs" / "course_config.json").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the RobotLocomotion-Race checkout. "
        "Set os.environ['COURSE_REPO_DIR'] to the repository path and rerun this cell."
    )


COURSE_REPO_DIR = find_course_repo()
BASE_DIR = COURSE_REPO_DIR / ".local_notebook_deps"
BASE_DIR.mkdir(parents=True, exist_ok=True)

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

MENAGERIE_REPO = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF = "1b86ece576591213e2b666ebf59508454200ca97"

PLAYGROUND_DIR = BASE_DIR / "mujoco_playground"
UNITREE_DIR = BASE_DIR / "unitree_mujoco"
MENAGERIE_DIR = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"


def run(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)


def github_archive_url(repo_url: str, ref: str) -> str:
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"


def download_repo_snapshot(repo_url: str, ref: str, target_dir: Path) -> None:
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [path for path in tmp_dir.iterdir() if path.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


def checkout_existing_repo(target_dir: Path, ref: str) -> None:
    try:
        run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git fetch failed for {target_dir}: {exc}. Trying local checkout.")
    run(["git", "-C", target_dir, "checkout", ref])


def ensure_pinned_repo(repo_url: str, ref: str, target_dir: Path) -> None:
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            checkout_existing_repo(target_dir, ref)
            return
        except subprocess.CalledProcessError as exc:
            print(f"[warn] local git checkout failed for {target_dir}: {exc}. Re-downloading snapshot.")
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        run(["git", "clone", repo_url, target_dir])
        checkout_existing_repo(target_dir, ref)
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git path failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)


print("course repo:", COURSE_REPO_DIR)
print("local dependency cache:", BASE_DIR)


## 2. Install packages and fetch server-local dependencies


In [ ]:
import shutil

if shutil.which("ffmpeg") is None:
    print("[local] ffmpeg is not on PATH; imageio-ffmpeg from requirements is enough for local tests.")

%cd {COURSE_REPO_DIR}
!python -m pip install -q -U pip setuptools wheel
!python -m pip uninstall -y playground || true

ensure_pinned_repo(PLAYGROUND_REPO, PLAYGROUND_REF, PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_pinned_repo(MENAGERIE_REPO, MENAGERIE_REF, MENAGERIE_DIR)

!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}
%cd {PLAYGROUND_DIR}
!python -m pip install -q -e .
%cd {COURSE_REPO_DIR}

if str(COURSE_REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(COURSE_REPO_DIR.resolve()))
if str(PLAYGROUND_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PLAYGROUND_DIR.resolve()))

import jax
import mujoco_playground

print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())
print("mujoco_playground imported from:", mujoco_playground.__file__)
expected_playground = str(PLAYGROUND_DIR.resolve())
if expected_playground not in str(Path(mujoco_playground.__file__).resolve()):
    raise RuntimeError(f"Expected mujoco_playground to be imported from {expected_playground}")


## 3. Read the assignment requirements

Skim the short specs before editing.


In [ ]:
%cd {COURSE_REPO_DIR}
!sed -n '1,180p' docs/assignment_requirements.md
!sed -n '1,140p' docs/controller_interface.md
!sed -n '1,120p' docs/high_level_optimization_guide.md


## 4. Copy Go2 assets


In [ ]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}


## 5. Inspect the low-level Go2 environment


In [ ]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py --stage-name stage_2


## 6. Read the important starter files


In [ ]:
!sed -n '1,180p' go2_pg_env/joystick.py
!sed -n '1,220p' go2_pg_env/track.py
!sed -n '1,220p' track_bonus/controller_interface.py
!sed -n '1,220p' track_bonus/planner.py
!sed -n '1,220p' run_track_bonus.py


## 7. Define a server-friendly low-level training config

This config is smaller than the original Colab starter so the dry run and early experiments are manageable. Increase the step counts when you are ready for a serious GPU training run.


In [ ]:
import json

runtime_config = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [512, 256, 128],
    "value_hidden_layer_sizes": [512, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 10_000_000,
}

config_path = COURSE_REPO_DIR / "configs" / "server_runtime_config.json"
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config = json.loads(base_config_path.read_text())
base_config["runtime_overrides"] = runtime_config
config_path.write_text(json.dumps(base_config, indent=2))
print("wrote", config_path)


## 8. Dry-run training config


In [ ]:
!python train.py --config configs/server_runtime_config.json --dry-run


In [ ]:
# Experiment cell 1: train only the forward-motion low-level policy, then save a forward video.
from pathlib import Path
import json
import subprocess

%cd {COURSE_REPO_DIR}


def run_cmd(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)


def write_demo_config(base_config_path: Path, output_config_path: Path, command, *, segment_seconds=2.5, repeats=4):
    cfg = json.loads(Path(base_config_path).read_text())
    cfg["demo_rollout"] = {
        "segment_seconds": float(segment_seconds),
        "segments": [list(map(float, command)) for _ in range(int(repeats))],
    }
    output_config_path.parent.mkdir(parents=True, exist_ok=True)
    output_config_path.write_text(json.dumps(cfg, indent=2))
    return output_config_path


SERVER_CONFIG = COURSE_REPO_DIR / "configs" / "server_runtime_config.json"
if not SERVER_CONFIG.exists():
    raise FileNotFoundError("Run Step 7 first so configs/server_runtime_config.json exists.")

EXPERIMENT_DIR = COURSE_REPO_DIR / "artifacts" / "experiments"
STAGE1_DIR = EXPERIMENT_DIR / "stage1_forward_motion_reward_v4"
STAGE1_CHECKPOINT_DIR = STAGE1_DIR / "best_checkpoint"
STAGE1_VIDEO_DIR = EXPERIMENT_DIR / "videos" / "stage1_forward_reward_v4"
STAGE1_FORWARD_CONFIG = EXPERIMENT_DIR / "configs" / "stage1_forward_demo.json"

run_cmd([
    "python", "train.py",
    "--config", SERVER_CONFIG,
    "--stage", "stage_1",
    "--output-dir", STAGE1_DIR,
])

write_demo_config(SERVER_CONFIG, STAGE1_FORWARD_CONFIG, [0.6, 0.0, 0.0], segment_seconds=2.5, repeats=4)
run_cmd([
    "python", "test_policy.py",
    "--config", STAGE1_FORWARD_CONFIG,
    "--checkpoint-dir", STAGE1_CHECKPOINT_DIR,
    "--output-dir", STAGE1_VIDEO_DIR,
    "--stage-name", "stage_1",
    "--render-width", 960,
    "--render-height", 540,
])

print("stage 1 checkpoint:", STAGE1_CHECKPOINT_DIR)
print("stage 1 forward video:", STAGE1_VIDEO_DIR / "demo.mp4")


In [ ]:
# Experiment cell 2: continue with the adaptive command curriculum, then save diagnostic videos.
from pathlib import Path
import json
import subprocess

%cd {COURSE_REPO_DIR}


def run_cmd(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)


def write_demo_config(base_config_path: Path, output_config_path: Path, command, *, segment_seconds=2.5, repeats=4):
    cfg = json.loads(Path(base_config_path).read_text())
    cfg["demo_rollout"] = {
        "segment_seconds": float(segment_seconds),
        "segments": [list(map(float, command)) for _ in range(int(repeats))],
    }
    output_config_path.parent.mkdir(parents=True, exist_ok=True)
    output_config_path.write_text(json.dumps(cfg, indent=2))
    return output_config_path


SERVER_CONFIG = COURSE_REPO_DIR / "configs" / "server_runtime_config.json"
if not SERVER_CONFIG.exists():
    raise FileNotFoundError("Run Step 7 first so configs/server_runtime_config.json exists.")

EXPERIMENT_DIR = COURSE_REPO_DIR / "artifacts" / "experiments"
STAGE1_CHECKPOINT_DIR = EXPERIMENT_DIR / "stage1_forward_motion_reward_v4" / "best_checkpoint"
if not STAGE1_CHECKPOINT_DIR.exists():
    raise FileNotFoundError("Run Experiment cell 1 first so the stage-1 checkpoint exists.")

STAGE2_DIR = EXPERIMENT_DIR / "stage2_adaptive_curriculum_reward_v5"
STAGE2_CHECKPOINT_DIR = STAGE2_DIR / "best_checkpoint"

run_cmd([
    "python", "train.py",
    "--config", SERVER_CONFIG,
    "--stage", "stage_2",
    "--restore-checkpoint-dir", STAGE1_CHECKPOINT_DIR,
    "--output-dir", STAGE2_DIR,
])

video_commands = {
    "forward": [0.8, 0.0, 0.0],
    "backward": [-0.4, 0.0, 0.0],
    "right": [0.0, -0.3, 0.0],
    "left": [0.0, 0.3, 0.0],
    "forward_turn_left": [0.6, 0.0, 0.5],
    "forward_turn_right": [0.6, 0.0, -0.5],
}

video_paths = {}
for name, command in video_commands.items():
    demo_config = write_demo_config(
        SERVER_CONFIG,
        EXPERIMENT_DIR / "configs" / f"stage2_{name}_demo.json",
        command,
        segment_seconds=2.5,
        repeats=4,
    )
    output_dir = EXPERIMENT_DIR / "videos" / f"stage2_reward_v5_{name}"
    run_cmd([
        "python", "test_policy.py",
        "--config", demo_config,
        "--checkpoint-dir", STAGE2_CHECKPOINT_DIR,
        "--output-dir", output_dir,
        "--stage-name", "stage_2",
        "--render-width", 960,
        "--render-height", 540,
    ])
    video_paths[name] = output_dir / "demo.mp4"

print("stage 2 checkpoint:", STAGE2_CHECKPOINT_DIR)
print("videos:")
for name, path in video_paths.items():
    print(f"  {name}: {path}")
print("Note: backward is intentionally outside the current positive-vx curriculum bins.")


## 9. Train or reuse a low-level checkpoint

Use a newly trained checkpoint or point `CHECKPOINT_DIR` to a HW1 `best_checkpoint`.


In [ ]:
# Uncomment the command below to train a new low-level policy.
# !python train.py \
#   --config configs/server_runtime_config.json \
#   --stage both \
#   --output-dir artifacts/low_level_train

# Or set CHECKPOINT_DIR to an existing low-level best_checkpoint.
preferred_checkpoint = COURSE_REPO_DIR / "artifacts" / "experiments" / "stage2_adaptive_curriculum_reward_v5" / "best_checkpoint"
fallback_checkpoint = COURSE_REPO_DIR / "artifacts" / "experiments" / "stage2_adaptive_curriculum_reward_v4" / "best_checkpoint"
CHECKPOINT_DIR = preferred_checkpoint if preferred_checkpoint.exists() else fallback_checkpoint
PLANNER_CONFIG = COURSE_REPO_DIR / "configs" / "starter_planner.json"
print("checkpoint path:", CHECKPOINT_DIR)
print("planner config:", PLANNER_CONFIG)
print("checkpoint exists:", CHECKPOINT_DIR.exists())


## 10. Run single-policy track evaluation

The smoke command is short and writes to `track_eval_smoke`. For submission, run the full command into `track_eval`.


In [ ]:
TRACK_EVAL_SMOKE_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval_smoke"
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

if CHECKPOINT_DIR.exists():
    print("Found checkpoint. Running short no-render track smoke eval...")
    !python run_track_bonus.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --planner-config {PLANNER_CONFIG} \
      --config configs/server_runtime_config.json \
      --output-dir {TRACK_EVAL_SMOKE_DIR} \
      --entry-name {TEAM_NAME} \
      --duration-seconds 5 \
      --no-render
else:
    print("Skipping track eval: CHECKPOINT_DIR does not exist yet.")
    print("Train a policy above or set CHECKPOINT_DIR to your HW1 best_checkpoint, then rerun this cell.")

# Full evaluation with video, after the smoke run works:
# !python run_track_bonus.py \
#   --checkpoint-dir {CHECKPOINT_DIR} \
#   --planner-config {PLANNER_CONFIG} \
#   --config configs/server_runtime_config.json \
#   --output-dir {TRACK_EVAL_DIR} \
#   --entry-name {TEAM_NAME} \
#   --render-every 10 \
#   --render-fps 5


## 11. Train your high-level planner

Train a residual MLP planner that wraps the analytic baseline and learns command corrections while keeping the same 5D -> [vx, vy, yaw_rate] interface.

Do not change the track geometry. The evaluator uses the fixed official oval for reset, scoring, and rendering.

A valid learned planner has trained parameters, such as MLP weights. Store those weights in your submission and load them from `StarterTrackPlanner.load(planner_config)`.


In [ ]:
HIGHLEVEL_DIR = COURSE_REPO_DIR / "artifacts" / "residual_planner_train_v2"
INIT_PLANNER_CONFIG = COURSE_REPO_DIR / "artifacts" / "residual_planner_train" / "planner_config.json"
INIT_ARGS = f"--init-planner-config {INIT_PLANNER_CONFIG}" if INIT_PLANNER_CONFIG.exists() else ""

if CHECKPOINT_DIR.exists():
    !python train_residual_planner.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --base-planner-config configs/starter_planner.json \
      --config configs/server_runtime_config.json \
      --output-dir {HIGHLEVEL_DIR} \
      --iterations 24 \
      --population 32 \
      --eval-seconds 300 \
      --start-samples 1 \
      --sigma 0.025 \
      --min-sigma 0.004 \
      --base-speed 0.55 \
      --residual-vx-scale 0.65 \
      --residual-vy-scale 0.22 \
      --residual-yaw-scale 0.65 \
      --target-progress-speed 0.72 \
      --target-finish-seconds 260 \
      {INIT_ARGS}
    PLANNER_CONFIG = HIGHLEVEL_DIR / "planner_config.json"
    print("learned residual planner config:", PLANNER_CONFIG)
else:
    print("Skipping residual planner training: CHECKPOINT_DIR does not exist yet.")


## 12. Create submission metadata


In [ ]:
import json
submission = {
    "team_name": TEAM_NAME,
    "track2_option": "leaderboard",
    "checkpoint_dir": "best_checkpoint",
    "planner_config": "planner_config.json",
    "planner_code": "track_bonus/planner.py",
    "planner_weights": "planner_weights.npz, if used",
    "high_level_planner_type": "learned",
    "track_eval": "track_eval/results.json",
    "notes": "Briefly describe your low-level training, learned high-level planner, and failed ideas."
}
(COURSE_REPO_DIR / "submission.json").write_text(json.dumps(submission, indent=2))
print((COURSE_REPO_DIR / "submission.json").read_text())


## 13. Final local checklist

This only checks local paths. If `track_eval/results.json` is missing, run the full evaluation command in Step 10.


In [ ]:
from pathlib import Path
expected = {
    "checkpoint": CHECKPOINT_DIR,
    "planner_config": PLANNER_CONFIG,
    "submission_json": COURSE_REPO_DIR / "submission.json",
    "track_eval_results": TRACK_EVAL_DIR / "results.json",
}
for label, path in expected.items():
    print(label, path, "OK" if path.exists() else "MISSING")
print("Submit best_checkpoint/, planner_config.json, planner weights if used, changed planner code if any, submission.json, track_eval/results.json, and your short report.")
